# Single-Sensor Post-Stroke Gait Analysis (Colab Pipeline)

This Colab notebook implements a reproducible **dataset-first** pipeline matching the research proposal:
1. Dataset harmonization + standardization
2. Calibration-free gait event detection
3. VAE latent feature learning
4. SSL contrastive pretraining + supervised evaluation
5. Bias auditing (Integrated Gradients)

> Designed for Google Colab. Start from top and run each section in order.


## Section 0 — Colab Setup
Installs dependencies and prepares workspace.


In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn torch torchvision torchaudio pytorch-lightning einops tqdm
!pip -q install h5py pyarrow wfdb

import os
import math
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.signal import resample_poly, butter, filtfilt, find_peaks
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## Section 1 — Dataset Configuration
Provide download links / local paths for public datasets and map their schema.

Expected standardized channels after harmonization:
`['ax', 'ay', 'az', 'gx', 'gy', 'gz']`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ROOT = Path('/content/gait_ssl')
RAW = ROOT / 'raw'
PROC = ROOT / 'processed'
CKPT = ROOT / 'checkpoints'
for p in [RAW, PROC, CKPT]:
    p.mkdir(parents=True, exist_ok=True)

DATASETS = {
    'voisard': {
        'doi': 'https://doi.org/10.6084/m9.figshare.28806086',
        'local_path': RAW / 'voisard',
    },
    'felius': {
        'doi': 'https://doi.org/10.5281/zenodo.11045239',
        'local_path': RAW / 'felius',
    },
    'zhou': {
        'doi': 'https://doi.org/10.5281/zenodo.10534055',
        'local_path': RAW / 'zhou',
    },
}

TARGET_FS = 100
CHANNELS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
print('Configured datasets:', list(DATASETS))


## Section 2 — Harmonization Utilities (Phase 1)
- Polyphase resampling to 50/100 Hz
- Per-subject z-score normalization
- Outlier epoch filtering (|z| > 5)
- Clinical anchoring placeholders (10MWT, 6MWT, 2MWT)


In [ ]:
def polyphase_resample(x: np.ndarray, src_fs: int, tgt_fs: int = TARGET_FS) -> np.ndarray:
    if src_fs == tgt_fs:
        return x
    g = math.gcd(src_fs, tgt_fs)
    up, down = tgt_fs // g, src_fs // g
    return resample_poly(x, up=up, down=down, axis=0)


def zscore_per_subject(df: pd.DataFrame, subject_col='subject_id', cols=CHANNELS) -> pd.DataFrame:
    out = df.copy()
    for sid, idx in out.groupby(subject_col).groups.items():
        block = out.loc[idx, cols].astype(float)
        mu = block.mean(axis=0)
        sigma = block.std(axis=0).replace(0, 1.0)
        out.loc[idx, cols] = (block - mu) / sigma
    return out


def segment_epochs(arr: np.ndarray, epoch_len=512, overlap=0.5) -> np.ndarray:
    step = int(epoch_len * (1 - overlap))
    step = max(step, 1)
    chunks = []
    for i in range(0, len(arr) - epoch_len + 1, step):
        chunks.append(arr[i:i+epoch_len])
    return np.stack(chunks) if chunks else np.empty((0, epoch_len, arr.shape[-1]))


def filter_outlier_epochs(epochs: np.ndarray, z_thresh=5.0, max_ratio=0.01) -> np.ndarray:
    exceed_ratio = (np.abs(epochs) > z_thresh).mean(axis=(1, 2))
    keep = exceed_ratio <= max_ratio
    return epochs[keep]


def butter_bandpass(data: np.ndarray, fs=TARGET_FS, low=0.01, high=10.0, order=1) -> np.ndarray:
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data, axis=0)


## Section 3 — Calibration-Free Gait Events (Phase 2)
Includes yaw/gravity frame alignment and heuristic HS/TO extraction.


In [ ]:
def estimate_gravity(acc: np.ndarray, fs=TARGET_FS, window_s=1.5) -> np.ndarray:
    win = max(3, int(window_s * fs))
    pad = win // 2
    padded = np.pad(acc, ((pad, pad), (0,0)), mode='edge')
    out = np.zeros_like(acc)
    for i in range(len(acc)):
        out[i] = padded[i:i+win].mean(axis=0)
    return out


def align_to_gravity(acc: np.ndarray, gyro: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    g = estimate_gravity(acc)
    z = g / (np.linalg.norm(g, axis=1, keepdims=True) + 1e-8)
    # Approximate yaw-invariant projection: remove gravity component.
    acc_vert = np.sum(acc * z, axis=1, keepdims=True)
    acc_h = acc - acc_vert * z
    aligned_acc = np.concatenate([acc_h[:, :2], acc_vert], axis=1)
    aligned_gyro = gyro.copy()
    return aligned_acc, aligned_gyro


def detect_hs_to(acc_aligned: np.ndarray, gyro_aligned: np.ndarray, fs=TARGET_FS):
    vertical_acc = acc_aligned[:, 2]
    sagittal_gyro = gyro_aligned[:, 1]

    hs_idx, _ = find_peaks(vertical_acc, distance=max(1, int(0.25 * fs)))
    to_idx, _ = find_peaks(-sagittal_gyro, distance=max(1, int(0.25 * fs)))
    return hs_idx, to_idx


def validate_events(pred_idx, ref_idx, fs=TARGET_FS, tol_ms=50):
    tol = int((tol_ms / 1000.0) * fs)
    hits = 0
    used = set()
    for p in pred_idx:
        for j, r in enumerate(ref_idx):
            if j in used:
                continue
            if abs(int(p) - int(r)) <= tol:
                hits += 1
                used.add(j)
                break
    precision = hits / max(1, len(pred_idx))
    recall = hits / max(1, len(ref_idx))
    return {'hits': hits, 'precision': precision, 'recall': recall}


## Section 4 — VAE for Latent Features (Phase 3)
Input shape: `(batch, 512, 6)` with 12-dimensional latent bottleneck. Encoder flatten size is inferred dynamically for architecture safety.


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, in_channels=6, latent_dim=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 256, kernel_size=5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(256, 128, kernel_size=5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2), nn.ReLU(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 512)
            enc_out = self.encoder(dummy)
        self._enc_shape = tuple(enc_out.shape[1:])
        self.flat_dim = int(np.prod(self._enc_shape))

        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.flat_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose1d(128, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose1d(256, in_channels, kernel_size=4, stride=2, padding=1),
        )

    def encode(self, x):
        h = self.encoder(x).reshape(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(z.size(0), *self._enc_shape)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        xr = self.decode(z)
        return xr, mu, logvar


def vae_loss(xr, x, mu, logvar, beta=1.0):
    recon = F.mse_loss(xr, x)
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kld, recon, kld


In [ ]:
def train_vae(train_loader, epochs=20, lr=1e-3):
    model = ConvVAE().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for ep in range(1, epochs+1):
        model.train()
        ep_loss = ep_recon = ep_kld = 0.0
        n = 0
        for xb in train_loader:
            xb = xb.to(DEVICE).transpose(1,2)
            xr, mu, logvar = model(xb)
            loss, recon, kld = vae_loss(xr, xb, mu, logvar)
            opt.zero_grad()
            loss.backward()
            opt.step()
            bs = xb.size(0)
            n += bs
            ep_loss += loss.item()*bs
            ep_recon += recon.item()*bs
            ep_kld += kld.item()*bs
        hist.append({'epoch': ep, 'loss': ep_loss/n, 'mse': ep_recon/n, 'kld': ep_kld/n})
        print(hist[-1])
    return model, pd.DataFrame(hist)


## Section 5 — SSL Pretraining Scaffold (Phase 4)
Contrastive pretraining with NT-Xent and sensor-aware augmentations.


In [ ]:
def aug_axis_swap(x):
    p = np.random.permutation(3)
    y = x.copy()
    y[:, :3] = y[:, p]
    y[:, 3:] = y[:, 3 + p]
    return y


def aug_sensor_drift(x, slope_range=0.02):
    y = x.copy()
    t = np.linspace(0, 1, len(x))[:, None]
    s = np.random.uniform(-slope_range, slope_range, size=(1, x.shape[1])) * np.std(x, axis=0, keepdims=True)
    return y + t * s


def aug_jitter(x):
    sigma = random.choice([0.005, 0.01, 0.02])
    return x + np.random.randn(*x.shape) * sigma * np.std(x, axis=0, keepdims=True)


def augment_window(x):
    y = aug_axis_swap(x)
    y = aug_sensor_drift(y)
    y = aug_jitter(y)
    return y


class ProjectionHead(nn.Module):
    def __init__(self, in_dim=12, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, out_dim))

    def forward(self, z):
        return F.normalize(self.net(z), dim=-1)


def nt_xent(z1, z2, tau=0.07):
    b = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / tau
    mask = torch.eye(2*b, device=z.device).bool()
    sim.masked_fill_(mask, -1e9)
    target = torch.arange(b, device=z.device)
    target = torch.cat([target+b, target], dim=0)
    return F.cross_entropy(sim, target)


## Section 6 — Data Adapter Template
Adapt each dataset into a single standardized table with columns:
`subject_id, dataset_id, timestamp, ax, ay, az, gx, gy, gz, label(optional)`


In [ ]:
def load_dataset_template(dataset_id: str, source_path: Path) -> pd.DataFrame:
    # Replace this template with dataset-specific parsing.
    # Expected return shape: one row per sample.
    raise NotImplementedError(f'Implement parser for {dataset_id}: {source_path}')


def build_harmonized_table(dataset_cfg=DATASETS, target_fs=TARGET_FS) -> pd.DataFrame:
    rows = []
    for ds_id, cfg in dataset_cfg.items():
        src = Path(cfg['local_path'])
        df = load_dataset_template(ds_id, src)
        if df.empty:
            continue
        # Resample per subject recording
        for sid, block in df.groupby('subject_id'):
            arr = block[CHANNELS].to_numpy(dtype=np.float32)
            src_fs = int(block['sampling_rate_hz'].iloc[0])
            arr = polyphase_resample(arr, src_fs=src_fs, tgt_fs=target_fs)
            new_block = pd.DataFrame(arr, columns=CHANNELS)
            new_block['subject_id'] = sid
            new_block['dataset_id'] = ds_id
            rows.append(new_block)
    out = pd.concat(rows, ignore_index=True)
    out = zscore_per_subject(out)
    return out


## Section 7 — End-to-End Runner
Runs harmonization → filtering → VAE training placeholders.


In [ ]:
class EpochDataset(Dataset):
    def __init__(self, epochs: np.ndarray):
        self.x = torch.tensor(epochs, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i]


def run_pipeline(df: pd.DataFrame, fs=TARGET_FS):
    arr = df[CHANNELS].to_numpy(dtype=np.float32)
    arr = butter_bandpass(arr, fs=fs, low=0.01, high=10.0, order=1)
    epochs = segment_epochs(arr, epoch_len=512, overlap=0.5)
    epochs = filter_outlier_epochs(epochs, z_thresh=5.0)
    ds = EpochDataset(epochs)
    dl = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)
    vae, history = train_vae(dl, epochs=20, lr=1e-3)
    return vae, history, epochs


## Section 8 — Clinical and Bias Auditing Hooks
- Latent reliability: compute ICC externally by repeated sessions/raters.
- Bias audit: use Integrated Gradients to check heel-strike saliency focus.


In [ ]:
# Placeholder for Captum-based Integrated Gradients audit.
# !pip -q install captum
# from captum.attr import IntegratedGradients
# ig = IntegratedGradients(model)
# attrs, delta = ig.attribute(inputs, target=target_class, return_convergence_delta=True)
# Compare top-attribution timestamps against HS peaks in vertical acceleration.

print('Audit hooks ready. Add your classifier + labels before running IG.')


## Section 9 — Experiment Logging Checklist
Track and export:
- Resampling rate (50/100 Hz)
- Outlier rejection count
- Event matching within ±50 ms
- VAE MSE and KL divergence
- Reconstruction absolute error targets (<0.15G acc, <0.59 deg/s gyro)
- SSL downstream metrics and subgroup bias gaps
